In [ ]:
using Base
using BenchmarkTools
using DelimitedFiles
using Distributed
using Plots
using StatsBase

using Hwloc: num_physical_cores

rmprocs(workers());
addprocs(num_physical_cores());
println(nworkers())

@everywhere using DistributedArrays
@everywhere using Random: rand;

# Set-up multithreading

In [ ]:
# Set-up multithreading environment

# using IJulia
# installkernel("Julia (12 threads)", "--project=@.", env=Dict("JULIA_NUM_THREADS"=>"12"))

# Data

In [ ]:
probability_main = Dict([[0, 0.35], 
                         [1, 0.35],
                         [2, 0.35],
                         [3, 0.2],
                         [4, 0.2],
                         [5, 0.2],
                         [6, 0.2],
                         [7, 0.15],
                         [8, 0.1],
                         [9, 0.05],
                         [10, -1]]);

cost_data = Dict([[0, 10], 
                  [1, 10],
                  [2, 10],
                  [3, 20],
                  [4, 20],
                  [5, 20],
                  [6, 30],
                  [7, 30],
                  [8, 40],
                  [9, 50],
                  [10, 0]]);

# Essential functions

In [ ]:
# Monte Carlo simulation of leveling HEXA stats
function compute_hexa(n::Int, probability_main::Dict, cost_data::Dict, start::Array, goal_main::Int, reset_when::Int)
    stats = dzeros(Int32, (n, 3), workers(), [nworkers(), 1])
    cost = dzeros(Int32, (n, 1), workers(), [nworkers(), 1])

    stats_reset = dzeros(Int32, (n, 3), workers(), [nworkers(), 1])
    cost_reset = dzeros(Int32, (n, 1), workers(), [nworkers(), 1])
    
    stats_stop = dzeros(Int32, (n, 3), workers(), [nworkers(), 1])
    cost_stop = dzeros(Int32, (n, 1), workers(), [nworkers(), 1])

    @sync @distributed for k in 1:nworkers()
        stats_local = localpart(stats)
        cost_local = localpart(cost)

        stats_reset_local = localpart(stats_reset)
        cost_reset_local = localpart(cost_reset)

        stats_stop_local = localpart(stats_stop)
        cost_stop_local = localpart(cost_stop)

        stats_local[:, 1] .= start[1]
        stats_local[:, 2] .= start[2]
        stats_local[:, 3] .= start[3]

        for i in eachindex(cost_local)
            while sum(stats_local[i, 1:3]) < 20
                cost_local[i] += cost_data[stats_local[i, 1]]

                result = rand()
                
                if (result <= probability_main[stats_local[i, 1]]) && (stats_local[i, 1] < 10)
                    stats_local[i, 1] += 1
                else
                    if (stats_local[i, 2] < 10) && (stats_local[i, 3] < 10)
                        result_add = rand()
                        if result_add <= 0.5
                            stats_local[i, 2] += 1
                        else
                            stats_local[i, 3] += 1
                        end
                    elseif stats_local[i, 3] == 10
                        stats_local[i, 2] += 1
                    elseif stats_local[i, 2] == 10
                        stats_local[i, 3] += 1
                    end
                end

                if sum(stats_local[i, 1:3]) == reset_when
                    stats_reset_local[i, :] = stats_local[i, :]
                    cost_reset_local[i] = cost_local[i]
                end

                if (stats_stop_local[i, 1] == 0) && (20 - sum(stats_local[i, 1:3]) < goal_main - stats_local[i, 1])
                    stats_stop_local[i, :] = stats_local[i, :]
                    cost_stop_local[i] = cost_local[i]
                end
            end
        end
    end

    return stats, cost, stats_reset, cost_reset, stats_stop, cost_stop
end;


# Monte Carlo simulation of leveling HEXA stats until main goal is met
function compute_hexa_reset(n::Int, probability_main::Dict, cost_data::Dict, start::Array, goal_main::Int, reset_when::Int, reset_below_level::Int)
    stats = dzeros(Int32, (n, 3), workers(), [nworkers(), 1])
    cost = dzeros(Int32, (n, 1), workers(), [nworkers(), 1])
    resets = dzeros(Int32, (n, 1), workers(), [nworkers(), 1])

    @sync @distributed for k in 1:nworkers()
        stats_local = localpart(stats)
        cost_local = localpart(cost)
        resets_local = localpart(resets)

        stats_local[:, 1] .= start[1]
        stats_local[:, 2] .= start[2]
        stats_local[:, 3] .= start[3]

        for i in eachindex(cost_local)
            while (stats_local[i, 1] < goal_main)
                while (sum(stats_local[i, 1:3]) < 20) && (stats_local[i, 1] < goal_main)
                    cost_local[i] += cost_data[stats_local[i, 1]]

                    result = rand()
                    
                    if (result <= probability_main[stats_local[i, 1]]) && (stats_local[i, 1] < 10)
                        stats_local[i, 1] += 1
                    else
                        if (stats_local[i, 2] < 10) && (stats_local[i, 3] < 10)
                            result_add = rand()
                            if result_add <= 0.5
                                stats_local[i, 2] += 1
                            else
                                stats_local[i, 3] += 1
                            end
                        elseif stats_local[i, 3] == 10
                            stats_local[i, 2] += 1
                        elseif stats_local[i, 2] == 10
                            stats_local[i, 3] += 1
                        end
                    end

                    reset_check = (sum(stats_local[i, 1:3]) == reset_when) && (stats_local[i, 1] < reset_below_level)
                    reset_check += (20 - sum(stats_local[i, 1:3])) < (goal_main - stats_local[i, 1])
                    reset_check += (sum(stats_local[i, 1:3]) == 20) && (stats_local[i, 1] < goal_main)

                    if reset_check >= 1
                        stats_local[i, 1] = start[1]
                        stats_local[i, 2] = start[2]
                        stats_local[i, 3] = start[3]

                        resets_local[i] += 1
                    end
                end
            end
        end
    end

    return stats, cost, resets
end;


# Monte Carlo simulation of leveling HEXA stats until secondary goal is met
function compute_hexa_secondary_reset(n::Int, probability_main::Dict, cost_data::Dict, start::Array, goal_secondary::Int, reset_when::Int, reset_below_level::Int)
    stats = dzeros(Int32, (n, 3), workers(), [nworkers(), 1])
    cost = dzeros(Int32, (n, 1), workers(), [nworkers(), 1])
    resets = dzeros(Int32, (n, 1), workers(), [nworkers(), 1])

    @sync @distributed for k in 1:nworkers()
        stats_local = localpart(stats)
        cost_local = localpart(cost)
        resets_local = localpart(resets)

        stats_local[:, 1] .= start[1]
        stats_local[:, 2] .= start[2]
        stats_local[:, 3] .= start[3]

        for i in eachindex(cost_local)
            while (stats_local[i, 2] < goal_secondary) && (stats_local[i, 3] < goal_secondary)
                while (sum(stats_local[i, 1:3]) < 20) && (stats_local[i, 2] < goal_secondary) && (stats_local[i, 3] < goal_secondary)
                    cost_local[i] += cost_data[stats_local[i, 1]]

                    result = rand()
                    
                    if (result <= probability_main[stats_local[i, 1]]) && (stats_local[i, 1] < 10)
                        stats_local[i, 1] += 1
                    else
                        if (stats_local[i, 2] < 10) && (stats_local[i, 3] < 10)
                            result_add = rand()
                            if result_add <= 0.5
                                stats_local[i, 2] += 1
                            else
                                stats_local[i, 3] += 1
                            end
                        elseif stats_local[i, 3] == 10
                            stats_local[i, 2] += 1
                        elseif stats_local[i, 2] == 10
                            stats_local[i, 3] += 1
                        end
                    end

                    reset_check = (sum(stats_local[i, 1:3]) == reset_when) && (stats_local[i, 2] < reset_below_level) && (stats_local[i, 3] < reset_below_level)
                    reset_check += ((20 - sum(stats_local[i, 1:3])) < (goal_secondary - stats_local[i, 2])) && ((20 - sum(stats_local[i, 1:3])) < (goal_secondary - stats_local[i, 3]))
                    reset_check += (sum(stats_local[i, 1:3]) == 20) && (stats_local[i, 2] < goal_secondary) && (stats_local[i, 3] < goal_secondary)

                    if reset_check >= 1
                        stats_local[i, 1] = start[1]
                        stats_local[i, 2] = start[2]
                        stats_local[i, 3] = start[3]

                        resets_local[i] += 1
                    end
                end
            end
        end
    end

    return stats, cost, resets
end;


# Get quantiles
function process_data(q::Float64, goal::Int, folder_results::String)
    matrix_cost = zeros(Int32, 10, goal)
    matrix_resets = zeros(Int32, 10, goal)

    for reset_when in 10:19
        reset_below_level_minimum = maximum((0, goal - (20 - reset_when)))
        for reset_below_level in reset_below_level_minimum:(goal-1)
            cost = readdlm("./$(folder_results)/cost_$(goal)_$(reset_when)_$(reset_below_level).txt", Int32)
            resets = readdlm("./$(folder_results)/resets_$(goal)_$(reset_when)_$(reset_below_level).txt", Int32)

            matrix_cost[reset_when-9, reset_below_level+1] = trunc(Int32, quantile(vec(cost), q))
            matrix_resets[reset_when-9, reset_below_level+1] = trunc(Int32, quantile(vec(resets), q))
        end
    end

    return matrix_cost, matrix_resets
end;


# Get mean
function process_data(goal::Int, folder_results::String)
    matrix_cost = zeros(Int32, 10, goal)
    matrix_resets = zeros(Int32, 10, goal)

    for reset_when in 10:19
        reset_below_level_minimum = maximum((0, goal - (20 - reset_when)))
        for reset_below_level in reset_below_level_minimum:(goal-1)
            cost = readdlm("./$(folder_results)/cost_$(goal)_$(reset_when)_$(reset_below_level).txt", Int32)
            resets = readdlm("./$(folder_results)/resets_$(goal)_$(reset_when)_$(reset_below_level).txt", Int32)

            matrix_cost[reset_when-9, reset_below_level+1] = trunc(Int32, mean(cost))
            matrix_resets[reset_when-9, reset_below_level+1] = trunc(Int32, mean(resets))
        end
    end

    return matrix_cost, matrix_resets
end;

# Monte Carlo simulation of leveling HEXA stats

In [ ]:
# n = 10^6
# start = [0, 0, 0]
# goal_main = 10
# reset_when = 10

# stats_d, cost_d, stats_reset_d, cost_reset_d, stats_stop_d, cost_stop_d = compute_hexa(n, probability_main, cost_data, start, goal_main, reset_when);

# stats = Array(stats_d)
# cost = Array(cost_d)

# stats_reset = Array(stats_reset_d)
# cost_reset = Array(cost_reset_d)

# stats_stop = Array(stats_stop_d)
# cost_stop = Array(cost_stop_d)

# d_closeall()

# Monte Carlo simulation of leveling HEXA main stat until goal is met with varying reset levels

In [ ]:
# n_magnitude = 8
# start = [0, 0, 0]
# goal_main = 9
# folder_results = "results"

# if !isdir("./$(folder_results)")
#     mkdir("./$(folder_results)")
# end

# for reset_when in 10:19
#     reset_below_level_minimum = maximum((0, goal_main - (20 - reset_when)))
#     for reset_below_level in reset_below_level_minimum:(goal_main-1)
#         stats_d, cost_d, resets_d  = compute_hexa_reset(10^2, probability_main, cost_data, start, goal_main, reset_when, reset_below_level);

#         stats = Array(stats_d)
#         cost = Array(cost_d)
#         resets = Array(resets_d)

#         d_closeall()

#         resets_magnitude = trunc(Int, log10(mean(resets)))
#         println(goal_main, " ", reset_when, " ", reset_below_level, " ", resets_magnitude)

#         if n_magnitude > resets_magnitude
#             stats_d, cost_d, resets_d  = compute_hexa_reset(10^(n_magnitude - resets_magnitude), probability_main, cost_data, start, goal_main, reset_when, reset_below_level);

#             stats = Array(stats_d)
#             cost = Array(cost_d)
#             resets = Array(resets_d)

#             d_closeall()
#         end

#         writedlm("./$(folder_results)/stats_$(goal_main)_$(reset_when)_$(reset_below_level).txt", stats)
#         writedlm("./$(folder_results)/cost_$(goal_main)_$(reset_when)_$(reset_below_level).txt", cost)
#         writedlm("./$(folder_results)/resets_$(goal_main)_$(reset_when)_$(reset_below_level).txt", resets)
#     end
# end

# Monte Carlo simulation of leveling HEXA secondary stats until goal is met with varying reset levels

In [ ]:
n_magnitude = 8
start = [0, 0, 0]
goal_secondary = 10
folder_results = "results_secondary"

if !isdir("./$(folder_results)")
    mkdir("./$(folder_results)")
end

for reset_when in 10:19
    reset_below_level_minimum = maximum((0, goal_secondary - (20 - reset_when)))
    for reset_below_level in reset_below_level_minimum:(goal_secondary-1)
        stats_d, cost_d, resets_d  = compute_hexa_secondary_reset(10^2, probability_main, cost_data, start, goal_secondary, reset_when, reset_below_level);

        stats = Array(stats_d)
        cost = Array(cost_d)
        resets = Array(resets_d)

        d_closeall()

        resets_magnitude = trunc(Int, log10(mean(resets)))
        println(goal_secondary, " ", reset_when, " ", reset_below_level, " ", resets_magnitude)

        if n_magnitude > resets_magnitude
            stats_d, cost_d, resets_d  = compute_hexa_secondary_reset(10^(n_magnitude - resets_magnitude), probability_main, cost_data, start, goal_secondary, reset_when, reset_below_level);

            stats = Array(stats_d)
            cost = Array(cost_d)
            resets = Array(resets_d)

            d_closeall()
        end

        writedlm("./$(folder_results)/stats_$(goal_secondary)_$(reset_when)_$(reset_below_level).txt", stats)
        writedlm("./$(folder_results)/cost_$(goal_secondary)_$(reset_when)_$(reset_below_level).txt", cost)
        writedlm("./$(folder_results)/resets_$(goal_secondary)_$(reset_when)_$(reset_below_level).txt", resets)
    end
end

# Process data

In [ ]:
# goal = 10
# q = 0.99
# folder_results = "results_secondary"

# matrix_cost_mean, matrix_resets_mean = process_data(goal, folder_results);
# matrix_cost, matrix_resets = process_data(q, goal, folder_results);

# writedlm("./$(folder_results)/overview_mean_cost_$(goal).txt", matrix_cost_mean)
# writedlm("./$(folder_results)/overview_mean_resets_$(goal).txt", matrix_resets_mean)
# writedlm("./$(folder_results)/overview_$(q)_cost_$(goal).txt", matrix_cost)
# writedlm("./$(folder_results)/overview_$(q)_resets_$(goal).txt", matrix_resets)

In [ ]:
goal = 10
q = 0.5
folder_results = "results_secondary"
matrix_cost, matrix_resets = process_data(q, goal, folder_results);

writedlm("./$(folder_results)/overview_$(q)_cost_$(goal).txt", matrix_cost)
writedlm("./$(folder_results)/overview_$(q)_resets_$(goal).txt", matrix_resets)